<a href="https://colab.research.google.com/github/Zattyla/AIStudio-by-katty/blob/main/K-Studio%20AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💌 K-Studio AI (v2.0)
### *Studio for stems separation and voice conversion (UVR & RVC)*

Developed by: **Katty (@nwletter on dc)** 🫧
Powered by: `audio-separator` (UVR) & `RVC-Project` (Voice Conversion)

---
**System Status:** 🚀 **UVR + RVC Ready.** > **Requirement:** Use **T4 GPU** (`Runtime` > `Change runtime type`).  
**Workflow:** 1. Separate Stems ➜ 2. Download Voice Model ➜ 3. Convert Voice.

---
#### 🫶🏻 **Features:**
* **Elite Models:** Support for BS-Roformer, MDX, and Kim Vocal 2.
* **Blue Aesthetic:** Custom UI in Pastel Cyan & Lavender.
* **One-Click:** Automated environment setup and GPU acceleration.

## 📘 How to Use the Studio

### **Step 1: Setup**
* Run the **Install** cell (Cell 1). This prepares the engines for both separation and voice conversion. It takes about 3 minutes.

### **Step 2: Install Voices**
* Go to the **"3. Model Downloader"** tab inside the Blue Interface.
* Paste a direct link (Zip file) from Google Drive or HuggingFace.
* Give it a name (e.g., `Jimin_BTS`) and click **Install Voice**.

### **Step 3: Separate & Convert**
1. **Tab 1:** Upload your song and separate the vocals.
2. **Tab 2:** Upload the *Isolated Vocal* you just created.
3. Select your AI Voice (click **Refresh** if it doesn't show up).
4. Adjust **Pitch Shift** (-12 for Male-to-Female, +12 for Female-to-Male).
5. Click **Apply AI Voice** and use the ⬇️ **Download** button when finished.

In [ ]:
# @title 1. Install Full Environment (UVR + RVC Core)
import os

print("⏳ Installing libraries... Please wait (2-3 minutes).")
!pip install -q gradio audio-separator[gpu] fairseq parselmouth gdown

# Clone RVC core if it doesn't exist
RVC_PATH = "/content/RVC_Engine"
if not os.path.exists(RVC_PATH):
    !git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI {RVC_PATH} --quiet
    # Install specific RVC requirements
    !pip install -q -r {RVC_PATH}/requirements.txt

print("✨ Setup complete! Proceed to Step 2.")

In [ ]:
# @title 2. Launch UVR & RVC Studio
import gradio as gr
import os
import zipfile
import gdown
import subprocess
import traceback
from audio_separator.separator import Separator

# --- FOLDER CONFIGURATION ---
OUTPUT_DIR = "uvr_outputs"
MODEL_DIR = "uvr_models"
RVC_MODELS = "RVC_Models"
RVC_ENGINE = "/content/RVC_Engine"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RVC_MODELS, exist_ok=True)

# --- MODELS LIST ---
MODELS_UVR = ["BS-Roformer-Viperx-12.97.ckpt", "Kim_Vocal_2.onnx", "UVR-MDX-NET-Inst_HQ_4.onnx"]

def get_rvc_models():
    """Scans the RVC_Models folder for downloaded voices."""
    if not os.path.exists(RVC_MODELS): return ["No models found"]
    models = [d for d in os.listdir(RVC_MODELS) if os.path.isdir(os.path.join(RVC_MODELS, d))]
    return models if models else ["No models found"]

# --- DOWNLOADING LOGIC ---
def download_rvc_model(link, name):
    if not link or not name:
        return "❌ Error: Please provide both a Link and a Model Name."
    target_dir = os.path.join(RVC_MODELS, name)
    os.makedirs(target_dir, exist_ok=True)
    zip_path = os.path.join(target_dir, f"{name}.zip")
    try:
        if "drive.google.com" in link:
            gdown.download(link, zip_path, quiet=False, fuzzy=True)
        else:
            os.system(f"wget -q '{link}' -O '{zip_path}'")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(target_dir)
        os.remove(zip_path)
        return f"✅ Success! Model '{name}' installed. Go to Tab 2 and click 'Refresh Models'."
    except Exception as e:
        return f"❌ Download failed: {str(e)}"

# --- UVR SEPARATION LOGIC ---
def process_separation(audio_path, model, format_out):
    if not audio_path: raise gr.Error("Please upload an audio file!")
    try:
        gr.Info("Separating stems... Please wait.")
        sep = Separator(model_file_dir=MODEL_DIR, output_dir=OUTPUT_DIR, output_format=format_out)
        sep.load_model(model)
        files = sep.separate(audio_path)
        f0, f1 = os.path.join(OUTPUT_DIR, files[0]), os.path.join(OUTPUT_DIR, files[1])
        if "instrumental" in f0.lower(): return f1, f0
        return f0, f1
    except Exception as e: raise gr.Error(f"UVR Error: {str(e)}")

# --- RVC INFERENCE LOGIC (THE REAL ENGINE) ---
def process_rvc(input_audio, model_name, pitch_shift):
    if not input_audio or "No models" in model_name:
        raise gr.Error("Select a valid audio and a downloaded model!")

    gr.Info(f"Converting voice to {model_name}... This may take a minute.")

    # Locate .pth and .index files
    model_folder = os.path.join(RVC_MODELS, model_name)
    pth_file = next((os.path.join(model_folder, f) for f in os.listdir(model_folder) if f.endswith(".pth")), None)
    index_file = next((os.path.join(model_folder, f) for f in os.listdir(model_folder) if f.endswith(".index")), "")

    if not pth_file:
        raise gr.Error(f"No .pth file found in {model_name} folder!")

    output_path = os.path.join(OUTPUT_DIR, f"RVC_{model_name}_Result.wav")

    # RVC CLI Command
    command = [
        "python", f"{RVC_ENGINE}/infer_cli.py",
        "--f0up_key", str(int(pitch_shift)),
        "--input_path", input_audio,
        "--index_path", index_file,
        "--f0method", "rmvpe",
        "--pth_path", pth_file,
        "--opt_path", output_path,
        "--index_rate", "0.75"
    ]

    try:
        subprocess.run(command, check=True, capture_output=True, text=True)
        return output_path, gr.update(value=output_path, visible=True)
    except Exception as e:
        traceback.print_exc()
        raise gr.Error("RVC Engine Failure. Check Colab logs for details.")

# --- BLUE PASTEL THEME ---
blue_theme = gr.themes.Soft(primary_hue="cyan", secondary_hue="blue", neutral_hue="slate").set(
    body_background_fill="*neutral_50",
    button_primary_background_fill="*primary_200",
    button_primary_text_color="*primary_700",
    block_title_text_color="*primary_600"
)

# --- INTERFACE LAYOUT ---
with gr.Blocks(theme=blue_theme, title="AI Cover Studio") as app:
    gr.Markdown("# 🔗 AI Cover Studio (v2.0")

    with gr.Tabs():
        # TAB 1: UVR
        with gr.Tab("1. Stem Separation"):
            with gr.Row():
                with gr.Column():
                    in_audio = gr.Audio(type="filepath", label="Step 1: Upload Song")
                    sel_uvr = gr.Dropdown(choices=MODELS_UVR, value=MODELS_UVR[0], label="UVR Model")
                    btn_sep = gr.Button("🚀 Separate Vocals", variant="primary")
                with gr.Column():
                    out_voc = gr.Audio(label="Isolated Vocals", interactive=False)
                    out_inst = gr.Audio(label="Instrumental", interactive=False)
            btn_sep.click(process_separation, inputs=[in_audio, sel_uvr, gr.State("WAV")], outputs=[out_voc, out_inst])

        # TAB 2: RVC
        with gr.Tab("2. Voice Conversion"):
            with gr.Row():
                with gr.Column():
                    in_rvc = gr.Audio(type="filepath", label="Step 2: Select Vocal Track")
                    sel_rvc = gr.Dropdown(choices=get_rvc_models(), label="Select AI Voice", value=get_rvc_models()[0])
                    btn_refresh = gr.Button("🔄 Refresh Models", variant="secondary", size="sm")
                    pitch = gr.Slider(minimum=-12, maximum=12, value=0, step=1, label="Pitch Shift")
                    btn_rvc = gr.Button("✨ Apply AI Voice", variant="primary")
                with gr.Column():
                    out_rvc = gr.Audio(label="Final Result", interactive=False)
                    btn_download = gr.DownloadButton("⬇️ Download Result", visible=False, variant="primary")

            btn_refresh.click(lambda: gr.Dropdown(choices=get_rvc_models()), outputs=sel_rvc)
            btn_rvc.click(process_rvc, inputs=[in_rvc, sel_rvc, pitch], outputs=[out_rvc, btn_download])

        # TAB 3: DOWNLOADER
        with gr.Tab("3. Model Downloader"):
            with gr.Row():
                with gr.Column():
                    link_input = gr.Textbox(label="Link (GDrive/Direct)", placeholder="Paste zip link here...")
                    name_input = gr.Textbox(label="Model Name", placeholder="e.g. Jimin_BTS")
                    btn_dl = gr.Button("⬇️ Install Voice", variant="primary")
                with gr.Column():
                    dl_status = gr.Textbox(label="Status", interactive=False)
            btn_dl.click(download_rvc_model, inputs=[link_input, name_input], outputs=[dl_status])

if __name__ == "__main__":
    app.launch(share=True, debug=True)

---
### 🔴 **Troubleshooting (Error Guide)**

#### **1. Error: "No .pth file found"**
* **Reason:** The zip link you used doesn't have a `.pth` file or it's inside too many subfolders.
* **Fix:** Ensure the Zip file contains the `.pth` and `.index` directly. Try a different model link.

#### **2. Interface is "Lagging" or Disconnected**
* **Reason:** Google Colab kills inactive sessions or your internet flickered.
* **Fix:** Re-run **Cell 2** to generate a new `.gradio.live` link. Your files in `uvr_outputs` will still be there.

#### **3. Pitch sounds robotic or "Off"**
* **Tip:** * Male to Female: `+12`
    * Female to Male: `-12`
    * Same Gender: `0`
* Always use the **RMVPE** method (it is the default in this studio) for the best results.

#### **4. "Out of Memory" (VRAM)**
* **Reason:** You are processing a very long song (10min+).
* **Fix:** Refresh the page or restart the Runtime to clear the GPU memory.